In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load Dataset

In [ ]:
from torchvision.datasets import CIFAR10
data = CIFAR10(root=r'/content/drive//MyDrive/DDPM/data',download=True,train=True)

100%|██████████| 170M/170M [16:51<00:00, 169kB/s]


In [ ]:
from torchvision.datasets import CelebA
data_test = CelebA(root="/content/drive/MyDrive/DDPM/data", download=True, split="test")

Downloading...
From (original): https://drive.google.com/uc?id=0B7EVK8r0v71pZjFTYXZWM3FlRnM
From (redirected): https://drive.usercontent.google.com/download?id=0B7EVK8r0v71pZjFTYXZWM3FlRnM&confirm=t&uuid=a6c3b065-4c28-4333-b8a3-d508ea35239b
To: /content/drive/MyDrive/DDPM/data/celeba/img_align_celeba.zip
100%|██████████| 1.44G/1.44G [01:18<00:00, 18.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=0B7EVK8r0v71pblRyaVFSWGxPY0U
To: /content/drive/MyDrive/DDPM/data/celeba/list_attr_celeba.txt
100%|██████████| 26.7M/26.7M [00:00<00:00, 53.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1_ee_0u7vcNLOfNLegJRHmolfH5ICW-XS
To: /content/drive/MyDrive/DDPM/data/celeba/identity_CelebA.txt
100%|██████████| 3.42M/3.42M [00:00<00:00, 4.79MB/s]
Downloading...
From: https://drive.google.com/uc?id=0B7EVK8r0v71pbThiMVRxWXZ4dU0
To: /content/drive/MyDrive/DDPM/data/celeba/list_bbox_celeba.txt
100%|██████████| 6.08M/6.08M [00:00<00:00, 28.4MB/s]
Downloading...
From: https://drive.googl

# Environment setup

In [17]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cpu
Torchvision: 0.26.0+cpu
CUDA available: False


In [18]:
!pip install -q numpy matplotlib tqdm torch-fidelity

In [19]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
IMAGE_SIZE = 32
BATCH_SIZE = 64
NUM_WORKERS = 2
VAL_RATIO = 0.10
TIMESTEPS = 1000

DATA_ROOT = "/content/drive//MyDrive/DDPM/data"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.manual_seed(SEED)

print("Device:", device)

Device: cpu


# Image transformations

In [26]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x * 2 - 1)
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x * 2 - 1)
])

# Load CIFAR-10 and create a reproducible split

In [21]:
train_source = datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=train_transform
)

val_source = datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=False,
    transform=val_transform
)

split_generator = torch.Generator().manual_seed(SEED)

indices = torch.randperm(
    len(train_source),
    generator=split_generator
)

val_size = int(len(train_source) * VAL_RATIO)

val_indices = indices[:val_size].tolist()
train_indices = indices[val_size:].tolist()

train_data = Subset(train_source, train_indices)
val_data = Subset(val_source, val_indices)

print("Training images:", len(train_data))
print("Validation images:", len(val_data))

Training images: 45000
Validation images: 5000


The original 10,000 test images remain untouched for final evaluation. Labels are ignored for an unconditional DDPM

In [22]:
PROJECT_DIR = Path("/content/drive//MyDrive/DDPM/data")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "seed": SEED,
        "train_indices": train_indices,
        "val_indices": val_indices,
    },
    PROJECT_DIR / "cifar10_split.pt"
)
# saves image indices—not the images or a trained model, keeps your experiments consistent

# Create DataLoaders

In [23]:
train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    generator=torch.Generator().manual_seed(SEED)
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)

images, labels = next(iter(train_loader))

print("Batch shape:", images.shape)
print("Pixel range:", images.min().item(), images.max().item())

Batch shape: torch.Size([64, 3, 32, 32])
Pixel range: -1.0 1.0


# Precompute the noise schedule

In [24]:
betas = torch.linspace(
    1e-4,
    0.02,
    TIMESTEPS,
    dtype=torch.float32,
    device=device
)

alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

alpha_bars_prev = torch.cat([
    torch.ones(1, device=device),
    alpha_bars[:-1]
])

sqrt_alpha_bars = torch.sqrt(alpha_bars)
sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)

sqrt_recip_alphas = torch.rsqrt(alphas)

posterior_variance = (
    betas
    * (1.0 - alpha_bars_prev)
    / (1.0 - alpha_bars)
)

print("Noise schedule ready.")

Noise schedule ready.


# Test forward diffusion

In [25]:
def add_noise(clean_images, t):
    noise = torch.randn_like(clean_images)

    signal_scale = sqrt_alpha_bars[t].view(-1, 1, 1, 1)
    noise_scale = sqrt_one_minus_alpha_bars[t].view(-1, 1, 1, 1)

    noisy_images = (
        signal_scale * clean_images
        + noise_scale * noise
    )

    return noisy_images, noise


images = images.to(device)

t = torch.randint(
    0,
    TIMESTEPS,
    (images.size(0),),
    device=device
)

noisy_images, target_noise = add_noise(images, t)

print("Noisy image shape:", noisy_images.shape)
print("Target noise shape:", target_noise.shape)

Noisy image shape: torch.Size([64, 3, 32, 32])
Target noise shape: torch.Size([64, 3, 32, 32])
